# Run 2 — Train on clean continuous text (20,000 → 30,000 steps) with Hugging Face backups

**Kaggle settings:** Accelerator **GPU T4** · Internet **On** · Secret **HF_TOKEN** attached · dataset **qwenlab_v3_modular** attached. Use **Save & Run All** (or run the cells in order; turn off laptop sleep).

**Two ways to run (cell 1, `START_MODE`):**
- `continue_old` *(default)*: take your existing 20,000-step model (`Akzhai/Thesis_Scratch_1`, left untouched) and train **20,000 → 30,000** on clean continuous text (no Q&A). The learning-rate schedule **restarts** (the old run had already decayed to 2e-5, so a plain 30k plan would jump back to 2e-4) and the new steps use documents the old run never saw.
- `scratch`: random weights, **0 → 30,000**, pausing at 20,000 (`STOP_AT_STEP = 20000`) to export the 20k model, then `STOP_AT_STEP = None` to finish.

**Across sessions:** a checkpoint is uploaded to `Thesis_Modular_v2_Ckpt` every 500 steps (old copies are pruned safely). If a session ends, start a new one and **run the same cells again**: it restores the checkpoint and continues exactly.

In [1]:
# ===================== 1. SETTINGS (the only cell you edit) =====================
import time
T0 = globals().get("T0", time.time())    # one session clock (re-running this cell does not reset it)
DRY_RUN = False                          # True: 30-step rehearsal of save / upload / resume / export with a tiny model

START_MODE   = "continue_old"            # "continue_old" | "scratch"   (see the notes above)
STOP_AT_STEP = None                      # scratch mode: 20000 for the first phase, then None.  continue_old: None

HF_USER = "Akzhai"
CKPT_REPO_NAME  = "Thesis_Modular_v2_Ckpt"   # live checkpoint (progress.pt) for resuming across sessions
EXPORT_REPO_20K = "Thesis_Modular_v2_20k"    # hf_model of the 20,000-step model   (scratch mode)
EXPORT_REPO_30K = "Thesis_Modular_v2_30k"    # hf_model of the 30,000-step model
OLD_REPO        = "Akzhai/Thesis_Scratch_1"  # your existing 20,000-step run (read-only here)
OLD_REPO_STEP   = 20000                      # the checkpoint in OLD_REPO must be at this step
OLD_REPO_SKIP_DOCS = 150_000                 # documents the old run already used: skipped, so new steps see NEW text

# ---- tokenizer and datasets: edit only here to change language / dataset ----
TOKENIZER_CFG = {"name": "hf", "path": "Qwen/Qwen2.5-0.5B", "eos_token": "<|endoftext|>"}   # "hf" | "sentencepiece" | "bytes"
DATA_SOURCE_CFG = {"kind": "hf_dataset",         # "hf_dataset" | "text_files" (.txt .jsonl .csv .parquet)
                   "params": {"path": "HuggingFaceFW/fineweb-edu", "name": "sample-10BT", "split": "train", "text_field": "text"}}
OOD_EVAL_CFG = {"kind": "hf_dataset",            # out-of-domain perplexity set
                "params": {"path": "Salesforce/wikitext", "name": "wikitext-103-raw-v1", "split": "test", "text_field": "text"}}

# ---- model (Qwen2.5-0.5B by default; other presets for a bigger GPU or quick tests) ----
VOCAB_SIZE   = 151936                    # exact Qwen2.5 / your trained model.  None = size from the tokenizer (new language)
MODEL_PRESET = "qwen0.5b"                # "qwen0.5b" | "medium" | "small" | "tiny"
PRESETS = {
    "qwen0.5b": dict(hidden_size=896, intermediate_size=4864, num_layers=24, num_heads=14, num_kv_heads=2),  # 494M params
    "medium":   dict(hidden_size=768, intermediate_size=2048, num_layers=16, num_heads=12, num_kv_heads=4),
    "small":    dict(hidden_size=512, intermediate_size=1536, num_layers=12, num_heads=8,  num_kv_heads=2),
    "tiny":     dict(hidden_size=128, intermediate_size=384,  num_layers=2,  num_heads=4,  num_kv_heads=2),
}
PRESET_DEFAULTS = {                      # micro-batch x accumulation x 512 tokens = tokens per step
    "qwen0.5b": dict(micro_batch=1, grad_accum=16, lr=2e-4),     # 8,192 tokens/step, fits the 15 GB T4
    "medium":   dict(micro_batch=2, grad_accum=8,  lr=3e-4),
    "small":    dict(micro_batch=4, grad_accum=4,  lr=5e-4),
    "tiny":     dict(micro_batch=4, grad_accum=4,  lr=1e-3),
}
INIT_SCHEME, INIT_STD, INIT_TRUNCATE = "gpt2_scaled", 0.02, 3.0

# ---- schedule (30,000 steps in total) ----
TARGET_STEPS   = 30_000
SCHEDULE       = "wsd"                   # warmup -> stable -> decay
DECAY_FRACTION = 0.2                     # last 20% of the (remaining) steps decay to MIN_LR_RATIO x peak
MIN_LR_RATIO   = 0.1
WARMUP         = 500                     # scratch mode
CONTINUE_LR_FRACTION, CONTINUE_WARMUP = 0.5, 200   # continue_old: restart at half the original peak (1e-4); a reasoned default, not tuned
WEIGHT_DECAY   = 0.1
SEQ_LEN        = 512                     # training context window AND the hard ceiling for generation
MIN_DOC_CHARS  = 200
SEED           = 42

# ---- checkpoints, evaluation, time ----
TOTAL_BUDGET_MIN     = 600               # whole-session cap (Kaggle's limit is 12 h)
RESERVE_MIN          = 45                # kept for the final save + export + upload
SAVE_EVERY           = 500
KEEP_LAST            = 1                 # local snapshots (Kaggle disk limit is ~20 GB)
UPLOAD_EVERY_N_SAVES = 1                 # 1 = upload the checkpoint on every 500-step save (2 = every 1000 steps)
EVAL_EVERY, EVAL_BATCHES, LOG_EVERY = 500, 25, 10     # 25 batches x 4 windows = 100 windows per evaluation
FORCE_FRESH_START    = False

# ---- guarded generation (professor's guard-rail) ----
MAX_NEW_TOKENS, MAX_NEW_TOKENS_CAP, MAX_SENTENCES = 48, 64, 2
SAMPLE_PROMPTS = ["The capital of France is", "In science, the most important idea is", "Once upon a time"]
EXPORT_ANYWAY_TO = None                  # normally only steps 20,000 / 30,000 are exported; set a repo id to export another step

if DRY_RUN:
    START_MODE, STOP_AT_STEP, MODEL_PRESET, TARGET_STEPS, WARMUP = "scratch", None, "tiny", 30, 5
    SAVE_EVERY, UPLOAD_EVERY_N_SAVES, EVAL_EVERY, EVAL_BATCHES, LOG_EVERY = 10, 1, 10, 4, 5
    CKPT_REPO_NAME += "-dryrun"; EXPORT_REPO_20K += "-dryrun"; EXPORT_REPO_30K += "-dryrun"
    EXPORT_ANYWAY_TO = f"{HF_USER}/{EXPORT_REPO_30K}"

assert START_MODE in ("continue_old", "scratch"), "START_MODE must be 'continue_old' or 'scratch'"
SEED_FROM_OLD_REPO = OLD_REPO if START_MODE == "continue_old" else None
MICRO_BATCH, GRAD_ACCUM = PRESET_DEFAULTS[MODEL_PRESET]["micro_batch"], PRESET_DEFAULTS[MODEL_PRESET]["grad_accum"]
BASE_LR = PRESET_DEFAULTS[MODEL_PRESET]["lr"]

def minutes_used(): return (time.time() - T0) / 60.0
def session_budget_min(): return max(5.0, TOTAL_BUDGET_MIN - RESERVE_MIN - minutes_used())

from pathlib import Path
RUN_NAME   = "modular_run_dry" if DRY_RUN else "modular_run"
OUTPUT_DIR = Path("/kaggle/working") / RUN_NAME
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
FINAL_DIR  = Path("/kaggle/working") / (RUN_NAME + "_final")
for d in (CKPT_DIR, FINAL_DIR): d.mkdir(parents=True, exist_ok=True)
print(f"DRY_RUN={DRY_RUN} | mode={START_MODE} | preset={MODEL_PRESET} | target={TARGET_STEPS} steps (stop_at={STOP_AT_STEP}) | save every {SAVE_EVERY}")

DRY_RUN=False | mode=continue_old | preset=qwen0.5b | target=30000 steps (stop_at=None) | save every 500


In [2]:
# ===================== 2. INSTALL qwenlab (writable copy; /kaggle/input is read-only) =====================
import subprocess, shutil, sys, zipfile
from pathlib import Path

def _find(pattern):
    r = subprocess.run(["find", "/kaggle/input", "-iname", pattern], capture_output=True, text=True)
    return [p for p in r.stdout.strip().split("\n") if p]

def _pkg_roots(paths):
    return [Path(p).parent for p in paths if (Path(p).parent / "src" / "qwenlab").is_dir()]

roots = _pkg_roots(_find("pyproject.toml"))
if not roots:
    zips = _find("*.zip")
    assert zips, "No qwenlab code under /kaggle/input: attach the qwenlab_v3_modular dataset (Add Input)"
    unz = Path("/kaggle/working/qwenlab_unzipped"); unz.mkdir(exist_ok=True)
    for z in zips:
        zipfile.ZipFile(z).extractall(unz)
    roots = _pkg_roots([str(p) for p in unz.rglob("pyproject.toml")])
assert roots, "Found files but no folder with pyproject.toml next to src/qwenlab"

WORK_PKG = Path("/kaggle/working/qwenlab_src")
if WORK_PKG.exists():
    shutil.rmtree(WORK_PKG)
shutil.copytree(roots[0], WORK_PKG)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(WORK_PKG), "sentencepiece", "pyarrow"], check=True)
sys.path.insert(0, str(WORK_PKG / "src"))
print("qwenlab installed from", roots[0], "->", WORK_PKG)

qwenlab installed from /kaggle/input/datasets/akshayprabhakaran07/qwen-2-5v2/qwenlab -> /kaggle/working/qwenlab_src


In [3]:
# ===================== 3. TESTS (must pass before any GPU time is spent) =====================
result = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], cwd=WORK_PKG, capture_output=True, text=True)
print(result.stdout[-2500:]); print(result.stderr[-1000:])
assert result.returncode == 0, "Tests failed: do not start training until this passes"
print("All tests passed.")

........................................................................ [ 78%]
....................                                                     [100%]
92 passed in 47.40s


All tests passed.


In [4]:
# ===================== 4. LOGIN, CHECKPOINT RESTORE, RUN PLAN =====================
import json, gc, sys, math, shutil, hashlib, dataclasses, torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import qwenlab
assert tuple(int(x) for x in qwenlab.__version__.split(".")[:2]) >= (0, 3), (
    f"qwenlab {qwenlab.__version__} is too old for this notebook: upload the latest qwenlab_v3_modular.zip as a NEW VERSION of your "
    f"Kaggle dataset and re-attach it")
from qwenlab.config import ModelConfig, InitConfig, TrainConfig, TokenizerConfig, SourceConfig, PretrainDataConfig
from qwenlab.data.tokenizers import build_tokenizer
from qwenlab.models.factory import resolve_model_config
from qwenlab.hub import HubBackup

sys.last_traceback = sys.last_value = sys.last_type = None            # free GPU memory left by an earlier failed attempt
for _n in ("model", "trainer"):
    globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
import os; os.environ["HF_TOKEN"] = HF_TOKEN
login(HF_TOKEN)

CKPT_REPO_ID = f"{HF_USER}/{CKPT_REPO_NAME}"
REPO_20K_ID, REPO_30K_ID = f"{HF_USER}/{EXPORT_REPO_20K}", f"{HF_USER}/{EXPORT_REPO_30K}"
hub = HubBackup(CKPT_REPO_ID)                                        # every Hub call retries and never stops training
for rid in (CKPT_REPO_ID, REPO_20K_ID, REPO_30K_ID):
    hub._try(f"create {rid}", lambda rid=rid: hub.api.create_repo(rid, private=True, exist_ok=True))
hub._ready = True

tok_cfg   = TokenizerConfig(**TOKENIZER_CFG)
tokenizer = build_tokenizer(tok_cfg)
base_mcfg = ModelConfig(vocab_size=VOCAB_SIZE or len(tokenizer), max_seq_len=SEQ_LEN, **PRESETS[MODEL_PRESET])
cfg_model = resolve_model_config(base_mcfg, tokenizer, context_len=SEQ_LEN) if VOCAB_SIZE is None else base_mcfg
print(f"Model '{MODEL_PRESET}': {cfg_model.num_params()/1e6:.1f}M params | vocab={cfg_model.vocab_size} | context={cfg_model.max_seq_len}")

progress = CKPT_DIR / "progress.pt"
HISTORY_PATH, PLAN_PATH, METRICS_PATH = OUTPUT_DIR / "training_history.json", OUTPUT_DIR / "run_plan.json", OUTPUT_DIR / "metrics.jsonl"
ckpt_info, RESUME_SOURCE = {}, None

def inspect_checkpoint(path):
    """Reads step + shapes (memory-mapped) and checks they match the model preset."""
    try:
        try:
            ck = torch.load(path, map_location="cpu", mmap=True, weights_only=False)
        except Exception:
            ck = torch.load(path, map_location="cpu", weights_only=False)
        sd = ck["model_state_dict"]
        shape = tuple(sd["embed_tokens.weight"].shape)
        n_layers = len({k.split(".")[1] for k in sd if k.startswith("layers.")})
        ok = shape == (cfg_model.vocab_size, cfg_model.hidden_size) and n_layers == cfg_model.num_layers
        ckpt_info["step"] = ck.get("step")
        print(f"   checkpoint: step {ck.get('step')}, embedding {shape}, {n_layers} layers -> {'matches' if ok else 'DOES NOT match'} the preset")
        del ck, sd
        return ok
    except Exception as e:
        ckpt_info["step"] = None
        print("   could not read checkpoint:", repr(e)[:150]); return None

def fetch(label, repo_id, own):
    """Download progress.pt from a repo and vet it. own=True also restores plan/history/metrics of YOUR run
    (never from the old repo: its history has a different format)."""
    src = HubBackup(repo_id)
    if not src.restore_file("progress.pt", progress):
        return False
    if inspect_checkpoint(progress) and (ckpt_info.get("step") or 0) > 0:
        if own:
            for name, path in (("training_history.json", HISTORY_PATH), ("run_plan.json", PLAN_PATH), ("metrics.jsonl", METRICS_PATH)):
                src.restore_file(name, path)
        return True
    progress.unlink(missing_ok=True)
    return False

if FORCE_FRESH_START:
    print("FORCE_FRESH_START=True: ignoring existing checkpoints."); progress.unlink(missing_ok=True)
else:
    if progress.exists():
        if inspect_checkpoint(progress): RESUME_SOURCE = "own"
        else: progress.unlink(missing_ok=True)
    if RESUME_SOURCE is None and fetch("live checkpoint repo", CKPT_REPO_ID, own=True):
        RESUME_SOURCE = "own"
    if RESUME_SOURCE is None and SEED_FROM_OLD_REPO:
        print(f"Seeding from {SEED_FROM_OLD_REPO} (that repo stays untouched)...")
        if fetch("old repo", SEED_FROM_OLD_REPO, own=False):
            RESUME_SOURCE = "seed"
            assert ckpt_info["step"] == OLD_REPO_STEP, f"{SEED_FROM_OLD_REPO} is at step {ckpt_info['step']}, expected {OLD_REPO_STEP}"

RESUMING = progress.exists() and (ckpt_info.get("step") or 0) > 0
if START_MODE == "continue_old" and not RESUMING:
    raise RuntimeError(f"continue_old needs the checkpoint from {OLD_REPO} (or your own {CKPT_REPO_ID}) but none could be loaded. "
                       f"Check the HF_TOKEN secret and the repo name, or set START_MODE='scratch'.")
if not RESUMING or RESUME_SOURCE == "seed":
    for p in (HISTORY_PATH, PLAN_PATH, METRICS_PATH): p.unlink(missing_ok=True)     # a new run: no stale history

# ---- run plan: fixed when the run starts, then re-used by every later session ----
if RESUME_SOURCE == "own" and not PLAN_PATH.exists() and START_MODE == "continue_old":
    raise RuntimeError(f"Found your checkpoint but not run_plan.json in {CKPT_REPO_ID}. Without it the schedule would restart "
                       f"again at the wrong step. Restore run_plan.json into the repo (it is uploaded at the start of every run).")
if RESUME_SOURCE == "own" and PLAN_PATH.exists():
    plan = json.load(open(PLAN_PATH))
    if TARGET_STEPS != plan["max_steps"]:
        print(f"[plan] using the run plan's max_steps={plan['max_steps']}, not TARGET_STEPS={TARGET_STEPS} "
              f"(changing it mid-run would change the learning-rate curve)")
else:
    restart = START_MODE == "continue_old"
    plan = {"max_steps": TARGET_STEPS, "schedule": SCHEDULE, "decay_fraction": DECAY_FRACTION, "seq_len": SEQ_LEN,
            "preset": MODEL_PRESET, "start_mode": START_MODE,
            "schedule_start_step": int(ckpt_info.get("step") or 0) if restart else 0,
            "skip_docs": OLD_REPO_SKIP_DOCS if restart else 0,
            "lr": BASE_LR * CONTINUE_LR_FRACTION if restart else BASE_LR,
            "warmup": CONTINUE_WARMUP if restart else WARMUP}
json.dump(plan, open(PLAN_PATH, "w"), indent=2)
print("RUN PLAN:", json.dumps(plan))
print(f"{'RESUMING from step ' + str(ckpt_info.get('step')) + ' (' + RESUME_SOURCE + ')' if RESUMING else 'FRESH run: step 0, random weights'}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[hub] restored training_history.json


run_plan.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

[hub] restored run_plan.json


metrics.jsonl:   0%|          | 0.00/9.85k [00:00<?, ?B/s]

[hub] restored metrics.jsonl
RUN PLAN: {"max_steps": 30000, "schedule": "wsd", "decay_fraction": 0.2, "seq_len": 512, "preset": "qwen0.5b", "start_mode": "continue_old", "schedule_start_step": 20000, "skip_docs": 150000, "lr": 0.0001, "warmup": 200, "data_fingerprint": "6625797449c8"}
RESUMING from step 21000 (own)


In [5]:
# ===================== 5. BUILD THE MODEL & CHECK THE INITIALISATION =====================
from qwenlab.models.qwen import QwenForCausalLM, count_parameters
from qwenlab.init import apply_init

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda" or DRY_RUN, "No GPU found: set Accelerator to GPU in the notebook settings"
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
PRECISION = ("bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp16") if device == "cuda" else "fp32"
print("Precision:", PRECISION)

model = QwenForCausalLM(cfg_model)
print(f"Parameters: {count_parameters(model):,} | context window (max_seq_len): {cfg_model.max_seq_len}")
json.dump({k: v for k, v in dataclasses.asdict(cfg_model).items() if k != "head_dim"}, open(OUTPUT_DIR / "model_config.json", "w"), indent=2)

if RESUMING:
    print(f"Resuming: weights and optimizer from step {ckpt_info.get('step')} are loaded by the trainer.")
else:
    apply_init(model, InitConfig(scheme=INIT_SCHEME, std=INIT_STD, truncate_sigma=INIT_TRUNCATE, seed=SEED))
    emb = model.embed_tokens.weight.detach().float()
    most_common = int(torch.unique(emb.flatten()[:1_000_000], return_counts=True)[1].max().item())
    assert most_common <= 50, "Weights look repeated/constant!"
    x = torch.randint(0, cfg_model.vocab_size, (2, 64), generator=torch.Generator().manual_seed(0))
    with torch.no_grad():
        init_loss = model.eval()(x, labels=x)[1].item()
    model.train()
    print(f"Untrained loss {init_loss:.3f} (uniform guessing ln({cfg_model.vocab_size}) = {math.log(cfg_model.vocab_size):.3f}); init checks passed.")

GPU: Tesla T4 | 15.6 GB
Precision: fp16
Parameters: 494,032,768 | context window (max_seq_len): 512
Resuming: weights and optimizer from step 21000 are loaded by the trainer.


In [6]:
# ===================== 6. DATA: continuous text only (Q&A / chat documents are dropped) =====================
import numpy as np
from qwenlab.data.pretrain_data import build_pretrain_data, build_eval_windows, lm_make_batch
from qwenlab.pipeline import data_fingerprint

seg_steps = plan["max_steps"] - plan["schedule_start_step"]            # steps still to train in this plan
train_tokens_need = int(seg_steps * GRAD_ACCUM * MICRO_BATCH * SEQ_LEN * 1.03) + SEQ_LEN + 1
val_tokens_need   = EVAL_BATCHES * 4 * SEQ_LEN + 1

src_cfg = SourceConfig(kind=DATA_SOURCE_CFG["kind"], params={**DATA_SOURCE_CFG["params"], "skip_docs": plan["skip_docs"]})
data_cfg = PretrainDataConfig(source=src_cfg, cache_dir="/kaggle/working/data_cache", seq_len=SEQ_LEN,
                              max_train_tokens=train_tokens_need, val_tokens=val_tokens_need, min_chars=MIN_DOC_CHARS)
t0 = time.time()
data = build_pretrain_data(data_cfg, tok_cfg, tokenizer)               # cached on disk; same text every session
lm_train, lm_val, data_stats = data.train, data.val, data.stats
assert len(lm_val) >= EVAL_BATCHES * 4, (f"only {len(lm_val)} validation windows, need {EVAL_BATCHES * 4}: "
                                         f"lower EVAL_BATCHES or use more text")
print(f"Data ready in {time.time()-t0:.0f}s: {len(lm_train):,} train windows | {len(lm_val):,} validation windows (skip_docs={plan['skip_docs']})")
print("Cleaning:", {k: v for k, v in data_stats.items() if k.startswith(("kept", "dropped"))})

# the rebuilt text must be identical to the text this run started on
fp = data_fingerprint(data)
if RESUME_SOURCE == "own" and plan.get("data_fingerprint") and plan["data_fingerprint"] != fp:
    raise RuntimeError(f"Training data changed since the run started ({plan['data_fingerprint']} -> {fp}); resuming would "
                       f"repeat or skip text. Use the same data settings or start a new run.")
plan["data_fingerprint"] = fp
json.dump(plan, open(PLAN_PATH, "w"), indent=2)
hub.upload_file(PLAN_PATH, "run_plan.json", "run plan")                 # on the Hub before the first checkpoint

ood_val = None
try:
    ood_val = build_eval_windows(dataclasses.replace(data_cfg, min_chars=80), tokenizer,
                                 SourceConfig(**OOD_EVAL_CFG), max_tokens=val_tokens_need)
    print(f"Out-of-domain windows ready: {len(ood_val):,}")
except Exception as e:
    print("Out-of-domain set unavailable, continuing without it:", repr(e)[:150])
print("\n--- sample training window ---\n", tokenizer.decode(lm_train[0][:70].tolist()), "...")

README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

[data] 20,000 docs, 20,845,157 tokens
[data] 40,000 docs, 41,618,334 tokens
[data] 60,000 docs, 62,838,345 tokens
[data] 80,000 docs, 83,759,298 tokens
[data] built 713bd716d095: {'kept': 80774, 'dropped_short': 0, 'dropped_qa_format': 153, 'dropped_wrong_script': 0, 'train_tokens': 84378406, 'val_tokens': 51259, 'fingerprint': '713bd716d095', 'dropped_qa_format_note': 'documents containing chat/FIM markers or Question:/Answer: layouts are excluded'}
Data ready in 304s: 164,801 train windows | 100 validation windows (skip_docs=150000)
Cleaning: {'kept': 80774, 'dropped_short': 0, 'dropped_qa_format': 153, 'dropped_wrong_script': 0, 'dropped_qa_format_note': 'documents containing chat/FIM markers or Question:/Answer: layouts are excluded'}


No files have been modified since last commit. Skipping to prevent empty commit.


[hub] uploaded run_plan.json


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Out-of-domain windows ready: 100

--- sample training window ---
 Chondroitin sulfate consists of repeating chains of molecules called glycosaminoglycans
(GAGs). Chondroitin sulfate is a major constituent of cartilage, providing structure, holding water and
nutrients, and allowing other molecules to move through cartilage—an important property,
as there is no blood supply to cartilage.
 ...


In [7]:
# ===================== 7. TRAINER, EVALUATION, GUARDED SAMPLES, HUB BACKUP =====================
from qwenlab.data.mixing import ResumableBatcher, make_cycler
from qwenlab.evaluation.perplexity import evaluate_lm, perplexity_from_loss
from qwenlab.evaluation.repetition import repeated_ngram_fraction
from qwenlab.inference.generate import generate_ids, generate_text
from qwenlab.training.trainer import Trainer

TIME_BUDGET_MIN = session_budget_min() if not DRY_RUN else min(session_budget_min(), 40.0)
train_cfg = TrainConfig(
    output_dir=str(OUTPUT_DIR), stage="pretrain", max_steps=plan["max_steps"], stop_at_step=STOP_AT_STEP,
    schedule_start_step=plan["schedule_start_step"], micro_batch_size=MICRO_BATCH, grad_accum_steps=GRAD_ACCUM,
    lr=plan["lr"], min_lr_ratio=MIN_LR_RATIO, warmup_steps=plan["warmup"], schedule=plan["schedule"],
    decay_fraction=plan["decay_fraction"], weight_decay=WEIGHT_DECAY, beta1=0.9, beta2=0.95, grad_clip=1.0,
    precision=PRECISION, log_every=LOG_EVERY, eval_every=EVAL_EVERY, eval_batches=EVAL_BATCHES,
    save_every=SAVE_EVERY, keep_last=KEEP_LAST, resume=True, time_budget_minutes=TIME_BUDGET_MIN, seed=SEED)
trainer = Trainer(model, train_cfg, device=device)
print(f"dtype={trainer.dtype} | scaler={trainer.use_scaler} | step {trainer.start_step} of {plan['max_steps']} | "
      f"lr peak {plan['lr']:.1e}, warmup {plan['warmup']} from step {plan['schedule_start_step']} | budget {TIME_BUDGET_MIN:.0f} min")

# batches: the data position is a pure function of the step, so a resumed session never repeats or skips text
mk_lm = lambda windows: lm_make_batch(windows, device=device)
seg_done = trainer.start_step - plan["schedule_start_step"]
train_fn = ResumableBatcher(lm_train, MICRO_BATCH, mk_lm, seed=SEED, start_batches=seg_done * GRAD_ACCUM)

EVAL_MICRO = 4
cut = lambda view, i: [view[i * EVAL_MICRO + j] for j in range(EVAL_MICRO)]
lm_val_fn  = make_cycler([mk_lm(cut(lm_val, i)) for i in range(EVAL_BATCHES)])
ood_val_fn = (make_cycler([mk_lm(cut(ood_val, i)) for i in range(EVAL_BATCHES)])
              if ood_val is not None and len(ood_val) >= EVAL_BATCHES * EVAL_MICRO else None)

history = json.load(open(HISTORY_PATH)) if (RESUME_SOURCE == "own" and HISTORY_PATH.exists()) else {"train": [], "val": [], "samples": []}
for _k in ("train", "val", "samples"):
    history[_k] = [r for r in history.get(_k, []) if r["step"] <= trainer.start_step]
history["data_stats"] = data_stats

def sample_guarded_text(step):
    """Text strictly inside the context window, at most 2 sentences."""
    texts = [f"{p} " + generate_text(trainer.model, tokenizer, p, context_len=SEQ_LEN, max_new_tokens=MAX_NEW_TOKENS,
                                     max_new_tokens_cap=MAX_NEW_TOKENS_CAP, temperature=0.0,
                                     max_sentences=MAX_SENTENCES, device=device).strip() for p in SAMPLE_PROMPTS]
    history["samples"].append({"step": step, "texts": texts})
    print(f"   --- guarded 2-sentence samples at step {step} ---")
    for t in texts: print("   |", t.replace("\n", " ")[:220])

def run_evals(step, in_res=None):
    """Loss AND perplexity: in-domain (FineWeb-Edu) and out-of-domain (WikiText-103)."""
    in_res = in_res or evaluate_lm(trainer.model, lm_val_fn, EVAL_BATCHES, device, trainer.dtype)
    rec = {"step": step, "lm_val": in_res["loss"], "lm_ppl": in_res["perplexity"], "tokens": in_res["tokens"]}
    ood_str = ""
    if ood_val_fn is not None:
        o = evaluate_lm(trainer.model, ood_val_fn, EVAL_BATCHES, device, trainer.dtype)
        rec.update(ood_val=o["loss"], ood_ppl=o["perplexity"])
        ood_str = f" | WikiText loss {rec['ood_val']:.4f} (ppl {rec['ood_ppl']:.2f})"
    history["val"] = [r for r in history["val"] if r["step"] != step] + [rec]
    print(f"   >>> step {step}: FineWeb loss {rec['lm_val']:.4f} (ppl {rec['lm_ppl']:.2f}){ood_str}")

def on_log(step, loss, lr, grad_norm):
    if not math.isfinite(loss):
        raise FloatingPointError(f"Loss became {loss} at step {step}: stopping to protect the last good checkpoint")
    el = time.time() - trainer.start_time
    history["train"].append({"step": step, "loss": loss, "lr": lr, "grad_norm": grad_norm})
    eta_h = ((STOP_AT_STEP or plan["max_steps"]) - step) * (el / max(step - trainer.start_step, 1)) / 3600.0
    print(f"step {step:5d}/{plan['max_steps']} | loss {loss:.4f} (ppl {perplexity_from_loss(loss):.1f}) | lr {lr:.2e} | "
          f"gnorm {grad_norm:.2f} | {el/60:.1f} min | ~{max(0.0, eta_h):.1f}h left")

def on_eval(step, val_loss):
    run_evals(step, trainer.last_eval)        # the trainer already measured the in-domain loss + perplexity
    sample_guarded_text(step)

# ---- Hugging Face backup on every save (never stops training; old copies pruned with guards) ----
upload_state = {"saves": 0, "uploaded_step": None}
def on_checkpoint(step, path, final):
    upload_state["saves"] += 1
    json.dump(history, open(HISTORY_PATH, "w"), indent=2)
    due = final or upload_state["saves"] % UPLOAD_EVERY_N_SAVES == 0
    if due and upload_state["uploaded_step"] != step:            # never upload the same step twice (6 GB each)
        t0 = time.time()
        if hub.upload_file(path, "progress.pt", f"checkpoint step {step}"):
            upload_state["uploaded_step"] = step
            for local, remote in [(HISTORY_PATH, "training_history.json"), (PLAN_PATH, "run_plan.json"),
                                  (OUTPUT_DIR / "model_config.json", "model_config.json"), (METRICS_PATH, "metrics.jsonl")]:
                hub.upload_file(local, remote, f"{remote} @ step {step}")
            print(f"[hf] checkpoint step {step} uploaded to {CKPT_REPO_ID} in {time.time()-t0:.0f}s | LFS cleanup: {hub.prune_stale_lfs()}")
        else:
            print(f"[hf] upload of step {step} FAILED; training continues, the local checkpoint is safe")
print(f"Backups armed: save every {SAVE_EVERY} steps, upload every {UPLOAD_EVERY_N_SAVES} save(s) to {CKPT_REPO_ID}.")

[resume] loaded /kaggle/working/modular_run/checkpoints/progress.pt, continuing from step 21000
dtype=torch.float16 | scaler=True | step 21000 of 30000 | lr peak 1.0e-04, warmup 200 from step 20000 | budget 548 min
Backups armed: save every 500 steps, upload every 1 save(s) to Akzhai/Thesis_Modular_v2_Ckpt.


In [8]:
# ===================== 8. TRAIN =====================
if not any(r["step"] == trainer.start_step for r in history["val"]):
    print(f"Measuring the starting model at step {trainer.start_step}:")
    run_evals(trainer.start_step)
    sample_guarded_text(trainer.start_step)

try:
    result = trainer.fit(train_fn, val_batch_fn=lm_val_fn, on_log=on_log, on_eval=on_eval, on_checkpoint=on_checkpoint)
except BaseException as e:
    print("\n!!! TRAINING STOPPED BY AN ERROR:", repr(e)[:300])
    try:
        torch.cuda.empty_cache()
        last = history["train"][-1]["step"] if history["train"] else trainer.start_step
        trainer.save_checkpoint(last, final=True)               # refuses to save NaN weights
        on_checkpoint(last, trainer._progress_path(), True)
        print(f"[emergency] checkpoint saved and uploaded at step {last}")
    except BaseException as e2:
        print("[emergency] could not save a new checkpoint (the last good one is unchanged):", repr(e2)[:200])
    raise

FINAL_STEP = result["final_step"]
if not history["val"] or history["val"][-1]["step"] != FINAL_STEP:
    run_evals(FINAL_STEP); sample_guarded_text(FINAL_STEP)
json.dump(history, open(HISTORY_PATH, "w"), indent=2)
planned = STOP_AT_STEP is not None and FINAL_STEP >= STOP_AT_STEP and FINAL_STEP < plan["max_steps"]
why = "planned stop reached" if planned else ("time budget reached" if result["stopped_early"] else "all planned steps completed")
print(f"\nStopped at step {FINAL_STEP} of {plan['max_steps']} ({why})")
if result["stopped_early"] and not planned:
    print(">>> Not finished: start a new session and run the same cells again; it resumes from the uploaded checkpoint.")

step 21010/30000 | loss 4.0417 (ppl 56.9) | lr 1.00e-04 | gnorm 1.22 | 0.5 min | ~7.2h left
step 21020/30000 | loss 4.0658 (ppl 58.3) | lr 1.00e-04 | gnorm 1.18 | 1.0 min | ~7.2h left
step 21030/30000 | loss 4.1880 (ppl 65.9) | lr 1.00e-04 | gnorm 2.10 | 1.5 min | ~7.2h left
step 21040/30000 | loss 4.0372 (ppl 56.7) | lr 1.00e-04 | gnorm 1.17 | 2.0 min | ~7.3h left
step 21050/30000 | loss 3.9453 (ppl 51.7) | lr 1.00e-04 | gnorm 1.17 | 2.5 min | ~7.3h left
step 21060/30000 | loss 4.2242 (ppl 68.3) | lr 1.00e-04 | gnorm 1.25 | 3.0 min | ~7.4h left
step 21070/30000 | loss 4.2171 (ppl 67.8) | lr 1.00e-04 | gnorm 1.34 | 3.5 min | ~7.4h left
step 21080/30000 | loss 4.1295 (ppl 62.1) | lr 1.00e-04 | gnorm 1.20 | 4.0 min | ~7.4h left
step 21090/30000 | loss 3.9492 (ppl 51.9) | lr 1.00e-04 | gnorm 1.23 | 4.5 min | ~7.5h left
step 21100/30000 | loss 4.2029 (ppl 66.9) | lr 1.00e-04 | gnorm 1.18 | 5.0 min | ~7.5h left
step 21110/30000 | loss 4.0635 (ppl 58.2) | lr 1.00e-04 | gnorm 1.23 | 5.6 min |

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[hub] uploaded progress.pt


No files have been modified since last commit. Skipping to prevent empty commit.


...

[hub] uploaded progress.pt


No files have been modified since last commit. Skipping to prevent empty commit.


[hub] uploaded training_history.json
[hub] uploaded run_plan.json


No files have been modified since last commit. Skipping to prevent empty commit.


[hub] uploaded model_config.json
[hub] uploaded metrics.jsonl
[hub]   stale: progress.pt 5.93 GB pushed 2026-10-25 10:49:13+00:00
[hf] checkpoint step 30000 uploaded to Akzhai/Thesis_Modular_v2_Ckpt in 48s | LFS cleanup: deleted 1 old copies (5.9 GB)

Stopped at step 30000 of 30000 (all planned steps completed)


In [9]:
# ===================== 9. EXPORT hf_model/ AND UPLOAD AS A SEPARATE MODEL REPO (steps 20,000 / 30,000 only) =====================
from qwenlab.export import export_hf

MILESTONES = {20000: REPO_20K_ID, 30000: REPO_30K_ID}
TARGET_MODEL_REPO_ID = EXPORT_ANYWAY_TO or MILESTONES.get(FINAL_STEP)
EXPORTED = False
if TARGET_MODEL_REPO_ID is None:
    print(f"Step {FINAL_STEP} is not a milestone (20,000 or 30,000): no model export. The checkpoint is backed up in {CKPT_REPO_ID}; "
          f"run the notebook again to continue.")
else:
    print(f"Exporting step {FINAL_STEP} -> {TARGET_MODEL_REPO_ID}")
    diff = export_hf(trainer.model, cfg_model, tokenizer, FINAL_DIR / "hf_model")        # reloads from disk and compares logits
    assert diff < 1e-3, f"Export does not match the trained model (max logit difference {diff})"
    for p in CKPT_DIR.glob("step_*.pt"):
        p.unlink()                                                                     # free disk
    for fname in ("model_config.json", "run_plan.json", "metrics.jsonl"):
        if (OUTPUT_DIR / fname).exists(): shutil.copy2(OUTPUT_DIR / fname, FINAL_DIR / fname)
    json.dump(history, open(FINAL_DIR / "training_history.json", "w"), indent=2)
    json.dump(dataclasses.asdict(train_cfg), open(FINAL_DIR / "train_config.json", "w"), indent=2)
    first, last = (history["val"][0], history["val"][-1]) if history["val"] else ({}, {})
    json.dump({"start_eval": first, "final_eval": last, "final_step": FINAL_STEP, "plan": plan, "data_stats": data_stats},
              open(FINAL_DIR / "results.json", "w"), indent=2)
    target = HubBackup(TARGET_MODEL_REPO_ID)
    target._try("create repo", lambda: target.api.create_repo(TARGET_MODEL_REPO_ID, private=True, exist_ok=True)); target._ready = True
    ok = target.upload_folder(FINAL_DIR / "hf_model", "hf_model", f"model @ step {FINAL_STEP}")
    for fname in ("results.json", "training_history.json", "train_config.json", "model_config.json", "run_plan.json", "metrics.jsonl"):
        target.upload_file(FINAL_DIR / fname, fname, f"{fname} @ step {FINAL_STEP}")
    EXPORTED = ok
    print(("SUCCESS: " if ok else "UPLOAD FAILED (re-run this cell): ") + f"https://huggingface.co/{TARGET_MODEL_REPO_ID}")
    if SEED_FROM_OLD_REPO:
        print(f"Your old 20,000-step model is untouched at https://huggingface.co/{SEED_FROM_OLD_REPO}")

Exporting step 30000 -> Akzhai/Thesis_Modular_v2_30k


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[export] hf_model written to /kaggle/working/modular_run_final/hf_model | max logit difference vs reloaded model: 7.82e-05
[hub] uploaded folder hf_model
[hub] uploaded results.json
[hub] uploaded training_history.json
[hub] uploaded train_config.json
[hub] uploaded model_config.json
[hub] uploaded run_plan.json
[hub] uploaded metrics.jsonl
SUCCESS: https://huggingface.co/Akzhai/Thesis_Modular_v2_30k
Your old 20,000-step model is untouched at https://huggingface.co/Akzhai/Thesis_Scratch_1


In [ ]:
# ===================== 10. VERIFY, PERPLEXITY TABLE, REPETITION DIAGNOSTIC =====================
from transformers import AutoModelForCausalLM

def load_hf(repo=None, sub=None, path=None):
    kw = dict(subfolder=sub) if sub else {}
    try:
        return AutoModelForCausalLM.from_pretrained(path or repo, dtype=torch.float32, **kw).eval()
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(path or repo, torch_dtype=torch.float32, **kw).eval()

if EXPORTED and not DRY_RUN:
    local_m, hub_m = load_hf(path=str(FINAL_DIR / "hf_model")), load_hf(TARGET_MODEL_REPO_ID, "hf_model")
    xx = torch.randint(0, cfg_model.vocab_size, (1, 24))
    with torch.no_grad():
        dd = (hub_m(input_ids=xx).logits - local_m(input_ids=xx).logits).abs().max().item()
    print(f"Hub model {TARGET_MODEL_REPO_ID} vs local export: max logit difference {dd:.1e}"); assert dd < 1e-4
    del local_m, hub_m; gc.collect()

if history["val"]:
    a, b = history["val"][0], history["val"][-1]
    print("\n" + "=" * 88)
    print(f"{'Continuous-text pretraining':38s} | {'Step %d (start)' % a['step']:>20s} | {'Step %d (final)' % b['step']:>20s}")
    print("-" * 88)
    print(f"{'FineWeb-Edu loss (in-domain)':38s} | {a['lm_val']:20.4f} | {b['lm_val']:20.4f}")
    print(f"{'FineWeb-Edu perplexity (lower=better)':38s} | {a['lm_ppl']:20.2f} | {b['lm_ppl']:20.2f}")
    if "ood_val" in a and "ood_val" in b:
        print(f"{'WikiText-103 loss (out-of-domain)':38s} | {a['ood_val']:20.4f} | {b['ood_val']:20.4f}")
        print(f"{'WikiText-103 perplexity (lower=better)':38s} | {a['ood_ppl']:20.2f} | {b['ood_ppl']:20.2f}")
    print("=" * 88)
    print(f"Cleaning: kept {data_stats['kept']:,} prose docs | dropped {data_stats['dropped_qa_format']:,} Q&A/chat docs")

# professor's check: is repetition caused by running past the training context window?
diag = QwenForCausalLM(dataclasses.replace(cfg_model, max_seq_len=SEQ_LEN + 150)).to(device).eval()
diag.load_state_dict(trainer.model.state_dict())
prompt_ids = tokenizer.encode("Once upon a time")
new_ids = generate_ids(diag, prompt_ids, context_len=SEQ_LEN + 150, max_new_tokens=SEQ_LEN + 100,
                       max_new_tokens_cap=10**9, eos_id=None, device=device)
cut_pos = max(SEQ_LEN - len(prompt_ids), 8)
rep_in, rep_out = repeated_ngram_fraction(new_ids[:cut_pos], 4), repeated_ngram_fraction(new_ids[cut_pos:], 4)
print(f"\n[Repetition diagnostic] 4-gram repetition inside the {SEQ_LEN}-token window: {100*rep_in:.1f}% | beyond it: {100*rep_out:.1f}%")
del diag; gc.collect(); torch.cuda.empty_cache() if device == "cuda" else None
json.dump({"repetition_inside_window_pct": 100 * rep_in, "repetition_beyond_window_pct": 100 * rep_out, "step": FINAL_STEP},
          open(OUTPUT_DIR / "repetition_diagnostic.json", "w"))
if EXPORTED:
    HubBackup(TARGET_MODEL_REPO_ID).upload_file(OUTPUT_DIR / "repetition_diagnostic.json", "repetition_diagnostic.json")
if history.get("samples"):
    s = history["samples"][-1]
    print(f"\nFinal guarded 2-sentence samples (step {s['step']}):")
    for t in s["texts"]: print(" |", t.replace("\n", " ")[:240])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

hf_model/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.98GB            

hf_model/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

Hub model Akzhai/Thesis_Modular_v2_30k vs local export: max logit difference 0.0e+00

Continuous-text pretraining            |   Step 20000 (start) |   Step 30000 (final)
----------------------------------------------------------------------------------------
FineWeb-Edu loss (in-domain)           |               4.2006 |               3.8491
FineWeb-Edu perplexity (lower=better)  |                66.73 |                46.95
WikiText-103 loss (out-of-domain)      |               4.8809 |               4.5096
WikiText-103 perplexity (lower=better) |               131.74 |                90.88
Cleaning: kept 80,774 prose docs | dropped 153 Q&A/chat docs

[Repetition diagnostic] 4-gram repetition inside the 512-token window: 94.3% | beyond it: 85.1%
[hub] uploaded repetition_diagnostic.json

Final guarded 2-sentence samples (step 30000):
 | The capital of France is the French capital of France. The capital of France is the French capital of France.
 | In science, the most important idea 